# YOLO Training — Highlighted Text Detection
Fine-tune YOLO26n on synthetic highlighted-text dataset (7 colour classes).

In [1]:
!pip install --upgrade --force-reinstall ultralytics
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -Uq Pillow==11.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.7/97.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.8/846.8 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57

In [2]:
import os, yaml

# Dataset is auto-mounted at /kaggle/input/highlighted-pdf-detection/
SRC = "/kaggle/input/datasets/frabbate/mark2text-real-v1/dataset_page_level"

# Fix data.yaml path to point directly to input mount (avoids copying 1.6 GB)
with open(f"{SRC}/data.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["path"] = SRC
with open("/kaggle/working/data.yaml", "w") as f:
    yaml.dump(cfg, f)

train_imgs = len(os.listdir(f"{SRC}/images/train"))
val_imgs = len(os.listdir(f"{SRC}/images/val"))
test_imgs = len(os.listdir(f"{SRC}/images/test"))
print(f"Train: {train_imgs}, Val: {val_imgs}, Test: {test_imgs}")

Train: 1598, Val: 337, Test: 262


In [3]:
import os, torch
import ultralytics
ultralytics.checks()
from ultralytics import YOLO

Ultralytics 8.4.112 🚀 Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7048.4/8062.4 GB disk)


In [4]:
model = YOLO("yolo26n.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=200,
    patience=20,
    imgsz=1024,
    batch=16,
    optimizer="auto",
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,
    scale=0.3,
    translate=0.05,
    fliplr=0.0,
    flipud=0.0,
    close_mosaic=10,
    device=[-1, -1],
    project="/kaggle/working/yolo_output",
    name="yolo26s_highlight_detection",
    exist_ok=True,
)



# Export trained weights
best_path = "/kaggle/working/yolo_output/highlight_detection/weights/best.pt"
last_path = "/kaggle/working/yolo_output/highlight_detection/weights/last.pt"

Searching for 2 idle GPUs with free memory >= 20.0% and free utilization >= 0.0%...
Selected idle CUDA devices [0, 1]
Ultralytics 8.4.112 🚀 Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=Fals

In [5]:
# Validate on test set
metrics = model.val(data="/kaggle/working/data.yaml", split="test")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")


Ultralytics 8.4.112 🚀 Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.0±2.1 ms, read: 27.4±13.8 MB/s, size: 4559.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/frabbate/mark2text-real-v1/dataset_page_level/labels/test... 262 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 262/262 33.0it/s 7.9s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/frabbate/mark2text-real-v1/dataset_page_level/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 1.0it/s 17.0s
                   all        262       5025      0.934       0.89      0.936      0.736
Speed: 2.6ms preprocess, 16.6ms inference, 0.0ms loss, 1.8m

In [6]:
import shutil
from ultralytics import YOLO

src = "/kaggle/input/datasets/frabbate/yolo26-highlighter-v1/HT_detector_v7.8.pt"
dst = "/kaggle/working/HT_detector_v7.8.pt"
shutil.copy(src, dst)

model = YOLO(dst)
model.export(
    format="onnx",
    opset=12,        # compatibilità con onnxruntime-web
    simplify=True,   # ottimizza il grafo (constant folding, operator fusion)
    half=True        # FP16: dimezza la dimensione dei pesi
)

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.112 🚀 Python-3.12.13 torch-2.6.0+cu124 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 13.3 GFLOPs

PyTorch: starting from '/kaggle/working/HT_detector_v7.8.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 300, 6) (5.2 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 439ms
 Downloaded onnxruntime
Prepared 2 packages in 348ms
Installed 2 packages in 17ms
 + onnxruntime==1.28.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 1.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting ex

/usr/local/lib/python3.12/dist-packages/torch/onnx/symbolic_opset9.py:5383: UserWarning: Exporting aten::index operator of advanced indexing in opset 12 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.94...
ONNX: converting to FP16...
ONNX: export success ✅ 3.9s, saved as '/kaggle/working/HT_detector_v7.8.onnx' (4.9 MB)

Export complete (4.8s)
Results saved to /kaggle/working/HT_detector_v7.8.onnx
Predict:         yolo predict task=detect model=/kaggle/working/HT_detector_v7.8.onnx imgsz=1024 quantize=16
Validate:        yolo val task=detect model=/kaggle/working/HT_detector_v7.8.onnx imgsz=1024 data=/kaggle/working/data.yaml quantize=16 
Visualize:       https://netron.app


'/kaggle/working/HT_detector_v7.8.onnx'